# Top-k Jaccard Robustness Table

% of (pair, instance) observations with Jaccard distance > k-specific baseline, for k ∈ {2, 3, 4, 5}.

Answers reviewer question: does churn collapse toward 0 when k increases?
B is `100 * count(distance > threshold) / (10 * N)`. Five complete runs are required. Method-averaged summaries below average these percentages, not distances.

K100 reuses the main-table inputs: German/Diabetes whole files; ACS/GMSC complete c0–c99 chunks. Alternate whole files are preserved but are not fallback inputs.


In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

import os, pickle, glob, re, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
from pairwise_aggregation import pairwise_distances, pairwise_exceedance
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import seaborn as sns

try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    pass

matplotlib.rcParams.update({
    'font.size': 12, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
})

from plot_violin import compute_jaccard_distance, compute_jaccard_baseline

DATASETS  = ['german', 'diabetes', 'acs', 'gmsc']
MODELS    = ['mlp', 'ftt']
METHODS   = ['random', 'kmeans', 'pred_stratified', 'cte']
K_VALUES  = [2, 3, 4, 5]

DATASET_LABELS = {'german': 'German', 'diabetes': 'Diabetes', 'acs': 'ACS', 'gmsc': 'GMSC'}
MODEL_LABELS   = {'mlp': 'MLP', 'ftt': 'FTT'}
METHOD_LABELS  = {'random': 'Random', 'kmeans': 'KMeans',
                  'pred_stratified': 'PredStrat', 'cte': 'CTE'}

N_BG_SEEDS = 5
K100_N_CHUNKS = {'german': 1, 'diabetes': 1, 'acs': 100, 'gmsc': 100}
N_FEATS    = {'german': 16, 'diabetes': 8, 'acs': 8, 'gmsc': 10, 'arrhythmia': 278}
RES_DIR    = 'results_tree'
FIG_DIR    = 'figures_b/background_topk_table'
os.makedirs(FIG_DIR, exist_ok=True)

PAIRS_10 = list(itertools.combinations(range(N_BG_SEEDS), 2))

print('Config loaded.')

In [ ]:
# Reuse the exact K100 input policy of background_table.ipynb.
# Missing chunks are not replaced with an alternate whole-file result version.
def load_sv(ds, model, method, s_idx):
    stem = f'{ds}_{model}_0_0_{method}_s{s_idx}'
    n_chunks = K100_N_CHUNKS[ds]
    paths = ([os.path.join(RES_DIR, f'{stem}_sv.pkl')] if n_chunks == 1 else
             [os.path.join(RES_DIR, f'{stem}_c{c}_sv.pkl') for c in range(n_chunks)])
    missing = [path for path in paths if not os.path.exists(path)]
    if missing:
        print(f'[INCOMPLETE] {ds}/{model}/{method}/seed{s_idx}: '
              f'missing {len(missing)}/{n_chunks} required files')
        return None
    arrays = []
    for path in paths:
        with open(path, 'rb') as handle:
            obj = pickle.load(handle)
        arrays.append(np.asarray(obj.values if hasattr(obj, 'values') else obj['values']))
    return np.concatenate(arrays, axis=0)

# Baseline: minimum expected Jaccard distance across the existing Mallows q sweep
baselines = {}   # baselines[k][ds]
for k in K_VALUES:
    baselines[k] = {}
    for ds in DATASETS:
        d = N_FEATS[ds]
        baselines[k][ds] = compute_jaccard_baseline(d=d, r=k)['min']
        print(f'  baseline k={k}, {ds} (d={d}): {baselines[k][ds]:.4f}')

print('Load helper and baselines ready.')

In [ ]:
# B: threshold every pair-instance distance first; denominator = 10*N.
table_data = {k: {ds: {mo: {} for mo in MODELS} for ds in DATASETS} for k in K_VALUES}
aggregation_records = []
for ds in DATASETS:
    for model in MODELS:
        sv_cache = {method: [load_sv(ds, model, method, s) for s in range(N_BG_SEEDS)]
                    for method in METHODS}
        for k in K_VALUES:
            for method in METHODS:
                record = dict(dataset=ds, model=model, method=method, k=k,
                              input_policy='main-table K100; no version fallback',
                              input_layout='whole' if K100_N_CHUNKS[ds] == 1 else 'chunks_c0_c99')
                try:
                    distances = pairwise_distances(
                        sv_cache[method], lambda a, b: compute_jaccard_distance(a, b, k=k))
                    stats = pairwise_exceedance(distances, baselines[k][ds])
                    table_data[k][ds][model][method] = stats['percent']
                    record.update(status='complete', **stats)
                except ValueError as error:
                    table_data[k][ds][model][method] = float('nan')
                    record.update(status='not_computed', reason=str(error))
                    print(f'[NOT COMPUTED] {ds}/{model}/{method}/k{k}: {error}')
                aggregation_records.append(record)
        print(f'{ds}/{model}: done')
pd.DataFrame(aggregation_records).to_csv(os.path.join(FIG_DIR, 'pair_exceedance_counts.csv'), index=False)
print('B aggregation complete; method-averaged summary is a separate statistic.')


In [ ]:
ROW_LABELS = [f'{DATASET_LABELS[ds]} / {MODEL_LABELS[mo]}'
              for ds in DATASETS for mo in MODELS]
COL_LABELS = [METHOD_LABELS[m] for m in METHODS]
row_bounds = [k * len(MODELS) for k in range(1, len(DATASETS))]  # dataset separators

def make_annot(mat):
    out = np.empty(mat.shape, dtype=object)
    for idx in np.ndindex(mat.shape):
        v = mat[idx]
        out[idx] = '' if np.isnan(v) else f'{v:.0f}'
    return out

fig, axes = plt.subplots(1, 4, figsize=(22, 7), sharey=True)
fig.suptitle('% of (pair, instance) observations with Jaccard distance > baseline\n'
             '(bg_size = 100, KernelSHAP)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, k in zip(axes, K_VALUES):
    mat = np.full((len(DATASETS) * len(MODELS), len(METHODS)), float('nan'))
    for i, ds in enumerate(DATASETS):
        for j, mo in enumerate(MODELS):
            for l, method in enumerate(METHODS):
                val = table_data[k][ds][mo].get(method, float('nan'))
                mat[i * len(MODELS) + j, l] = val

    df = pd.DataFrame(mat, index=ROW_LABELS, columns=COL_LABELS)
    sns.heatmap(df, ax=ax, vmin=0, vmax=100, cmap='YlOrRd',
                annot=make_annot(mat), fmt='', annot_kws={'size': 10},
                linewidths=0.3, linecolor='white',
                cbar=(k == K_VALUES[-1]),
                cbar_kws={'label': '% comparisons > baseline', 'shrink': 0.6})

    for rb in row_bounds:
        ax.axhline(rb, color='black', linewidth=1.5)

    ax.set_title(f'k = {k}\n(dataset-specific baseline)',
                 fontsize=12)
    ax.set_xlabel('Sampling Method', fontsize=12)
    ax.tick_params(axis='x', rotation=30, labelsize=10)
    ax.tick_params(axis='y', rotation=0, labelsize=10)
    if k != K_VALUES[0]:
        ax.set_ylabel('')

fig.tight_layout()
fname = os.path.join(FIG_DIR, 'topk_jaccard_heatmap.pdf')
fig.savefig(fname, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved: {fname}')

In [ ]:
# Separate descriptive summary: average B percentages across available methods.
# Not a distance mean and not one configuration's 10N prevalence.
print('Method-averaged B percentages (available methods only):')
print(' ' * 30, '  '.join(f'k={k:d}' for k in K_VALUES))
print('-' * 60)
for ds in DATASETS:
    for mo in MODELS:
        vals = []
        for k in K_VALUES:
            row_vals = [table_data[k][ds][mo].get(m, np.nan) for m in METHODS]
            row_vals = [v for v in row_vals if not np.isnan(v)]
            vals.append(np.mean(row_vals) if row_vals else np.nan)
        label = f'{DATASET_LABELS[ds]:10s}/{MODEL_LABELS[mo]:3s}'
        print(f'{label:30s}', '  '.join(f'{v:5.1f}%' if not np.isnan(v) else '  N/A ' for v in vals))
    print()